# FrameWeave Ovis2.5-9B — multi-seed train

Notebook **chỉ train**. Không eval HERBench. Run All train ba biến thể trên ba seed, rồi copy checkpoint lên Drive.

Biến thể: `global_only`, `local_only`, `hybrid`. `hybrid` là `RelJoinResidual(variant="Hybrid")` của V6, không viết lại. Ba biến thể trong một seed dùng chung lịch, loss và một backbone.

$$
\widetilde x_{ip} = x_{ip} + m_{ip}\,\chi_i\,a_{ip}\,c_{ip}\,u_{ip}
$$

- Seed train: **1, 100, 2027**. Khác seed thì khác init và khác lịch K / thứ tự lựa chọn.
- Split PerceptionTest dựng **một lần với seed 42** (454 câu / 128 video train, 67 câu / 19 video val). Seed 1, 100, 2027 không chia lại video.
- Backbone [`ATH-MaaS/Ovis2.5-9B`](https://huggingface.co/ATH-MaaS/Ovis2.5-9B) đóng băng, revision `5a1f9b995c98ae5b85a28ec89d88d74295be0ea7`. Adapter fp32. Thinking **off**. 448×448.
- Loss V6: $\mathrm{logsumexp}(\mathcal L_{\mathrm{cand}},\mathcal L_{\mathrm{corr}})-\log 2$. Chọn epoch theo validation $L$, patience 3, tối đa 8 epoch.
- Torch trong `/content/ovis_env` (Python 3.11 + torch 2.4.0 + flash-attn 2.7).

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `ovis/reljoin`, `ablation/`, `configs/` và `run_ablation.py` vào `/content`. Không upload file source.

Không upload manifest. Notebook tải PerceptionTest; `prepare` ghi split canonical vào `canonical_split/`. Model vào `/content/checkpoints/ovis2_5_9b`, video vào `/content/perceptiontest_lite`. HERBench là notebook eval.

Mỗi seed một output riêng: `/content/outputs/frameweave_ovis_multiseed/seed_<seed>/`. Drive: `/content/drive/MyDrive/frameweave_ovis_multiseed_outputs`.

Chín lần train không kịp một session. Run All lần sau copy ngược từ Drive và `--resume`. Smoke hai update chỉ chạy cho seed 1 khi seed đó chưa có `best/adapter.pt`.



## 1. Configuration

**Input:** none · **Output:** seeds, pins, paths


In [ ]:
# 1. Configuration — seeds 1, 100, 2027. Split stays canonical seed 42.
from __future__ import annotations
from pathlib import Path

SEEDS = [1, 100, 2027]
SPLIT_SEED = 42
VARIANT = "global_only,local_only,hybrid"
VARIANT_LIST = ["global_only", "local_only", "hybrid"]

ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

MODEL_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
TRAIN_DATASET_ID = "lmms-lab/LLaVA-Video-178K"
TRAIN_DATASET_REVISION = "6d8c562dc26d70042a0d9704d1cae58c94b89098"

VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
CANON = OUT_ROOT / "canonical_split"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
CONFIG_DIR = ROOT / "configs" / "multiseed"

print({
    "SEEDS": SEEDS,
    "SPLIT_SEED": SPLIT_SEED,
    "VARIANT": VARIANT,
    "OUT_ROOT": str(OUT_ROOT),
    "CANON": str(CANON),
    "backbone": MODEL_ID,
    "method": "FrameWeave-Ovis-Multiseed-Train",
})


## 2. Environment installation (`ovis_env`)

Official Ovis2.5-9B pins: `torch==2.4.0`, `transformers==4.51.3`, `flash-attn==2.7.0.post2`.
Stop if flash-attn install fails — the ViT hard-imports it.


In [ ]:
# 2. Bootstrap isolated ovis_env (Python 3.11 + torch 2.4.0 + flash-attn 2.7)
from __future__ import annotations
import os, subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "ovis_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs" / "frameweave_ovis_multiseed" / "logs" / "cell4_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc

sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "pyarrow==25.0.1", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging",
])
print("Installing flash-attn==2.7.0.post2 (required by Ovis ViT)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, flash_attn, pyarrow, numpy; "
    "print(torch.__version__); print(transformers.__version__); print(flash_attn.__version__); "
    "print('numpy', numpy.__version__); print('pyarrow', pyarrow.__version__); "
    "assert numpy.__version__.startswith('1.'), numpy.__version__; "
    "assert pyarrow.__version__ == '25.0.1', pyarrow.__version__; "
    "print(torch.__file__); print(transformers.__file__)"
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/ovis_env/" in line, f"Package leaked outside venv: {line}"
print("ovis_env ready:", PYTHON)
print("LOG:", LOG)


In [ ]:
# 2b. Extra pip (same packages as the V6 notebook). Writes a log so a failure is inspectable.
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
LOG_DIR = ROOT / "outputs" / "frameweave_ovis_multiseed" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / "cell5_pip.log"

if not PYTHON.exists():
    msg = (
        f"Missing {PYTHON}. Run the ovis_env bootstrap cell first "
        "(section 2), then re-run this cell.\n"
    )
    LOG.write_text(msg, encoding="utf-8")
    print(msg)
    raise FileNotFoundError(msg)

cmd = [
    str(PYTHON), "-m", "pip", "install",
    "--disable-pip-version-check",
    "--upgrade-strategy", "only-if-needed",
    "scikit-learn==1.3.2",
    "matplotlib==3.8.4",
    "numpy==1.25.0",
]
print("$", " ".join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_text = f"$ {' '.join(cmd)}\nexit={proc.returncode}\n\n{proc.stdout or ''}"
LOG.write_text(log_text, encoding="utf-8")
print(proc.stdout or "")
print("LOG:", LOG)

persist = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
if persist.parent.exists():
    try:
        persist.mkdir(parents=True, exist_ok=True)
        dest = persist / "logs" / LOG.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(log_text, encoding="utf-8")
        print("Copied log →", dest)
    except OSError as exc:
        print("Persist copy skipped:", exc)

if proc.returncode != 0:
    raise RuntimeError(f"cell 5 pip failed ({proc.returncode}). See {LOG}")
print("cell 5 pip OK")


## 3. Drive

Run All phục hồi `canonical_split` và từng `seed_*` từ Drive trước khi train.


In [ ]:
# 3. Mount Drive and restore per-seed trees before training.
from __future__ import annotations
import os
import shutil
from pathlib import Path

ROOT = Path("/content")
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
CANON = OUT_ROOT / "canonical_split"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    PERSIST = OUT_ROOT / "persist_local"
PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)
os.environ["OVIS_ABLATION_ROOT"] = str(ROOT)
os.environ["PYTHONUNBUFFERED"] = "1"

def restore(remote: Path, local: Path) -> None:
    if not remote.is_dir() or remote.resolve() == local.resolve():
        return
    if not any(remote.iterdir()):
        return
    local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(remote, local, dirs_exist_ok=True)
    print("Restored", remote, "→", local)

restore(PERSIST / "canonical_split", CANON)
for seed in SEEDS:
    restore(PERSIST / f"seed_{seed}", OUT_ROOT / f"seed_{seed}")

print({
    "RELJOIN_PERSIST": str(PERSIST),
    "OUT_ROOT": str(OUT_ROOT),
    "MODEL_DIR": str(MODEL_DIR),
    "SEEDS": list(SEEDS),
})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


link(REPO_DIR / "ovis" / "reljoin", ROOT / "reljoin")
link(REPO_DIR / "ovis_ablation" / "ablation", ROOT / "ablation")
link(REPO_DIR / "ovis_ablation" / "configs", ROOT / "configs")
link(REPO_DIR / "ovis_ablation" / "run_ablation.py", ROOT / "run_ablation.py")
print("FrameWeave source linked from", REPO_DIR / "ovis_ablation", "and", REPO_DIR / "ovis" / "reljoin")


In [ ]:
# 4. Verify FrameWeave sources. The 454/67 split is built later from PerceptionTest.
from __future__ import annotations
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_ablation.py",
    ROOT / "configs" / "ovis_multiseed.yaml",
    ROOT / "ablation" / "modules.py",
    ROOT / "ablation" / "train.py",
    ROOT / "ablation" / "schedule.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "losses.py",
    ROOT / "reljoin" / "data_perception.py",
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("sources OK", len(required))


## 5. Dataset / model download

Notebook train không tải HERBench. Ovis vào `/content/checkpoints/ovis2_5_9b`. PerceptionTest vào `/content/perceptiontest_lite`.


In [ ]:
# 5. Download Ovis2.5-9B and PerceptionTest. No HERBench in this notebook.
from __future__ import annotations
import os
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell cài ovis_env trước"
os.environ["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
MODEL_HF_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"

print("Downloading Ovis2.5-9B (~18 GB bf16)...")
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

print("Downloading PerceptionTest train shard...")
subprocess.run([str(PYTHON), "-c", f"""
import sys
from pathlib import Path
sys.path.insert(0, {str(ROOT)!r})
from reljoin.data_perception import download_perception_lite
ann, videos = download_perception_lite(Path('/content/perceptiontest_lite'))
print(ann)
print(videos)
"""], check=True)
print("Download step done")


## 6. Canonical split (seed 42) và unit test

`prepare` với seed 42 ghi 454/128 và 67/19. Unit test không phải số paper. Seed train không được dựng lại split này.


In [ ]:
# 6. Build the canonical seed-42 split once, then run unit tests.
from __future__ import annotations
import json
import os
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
CANON = OUT_ROOT / "canonical_split"
CONFIG_DIR = ROOT / "configs" / "multiseed"
assert PYTHON.exists(), "Chạy cell cài ovis_env trước"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("RELJOIN_ROOT", str(ROOT))
os.environ.setdefault("OVIS_ABLATION_ROOT", str(ROOT))

try:
    SPLIT_SEED
except NameError:
    SPLIT_SEED = 42

def _yaml_scalar(value):
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, float):
        if value.is_integer():
            return f"{value:.1f}"
        return format(value, ".12g")
    if isinstance(value, int):
        return str(value)
    if isinstance(value, str):
        if value == "":
            return '""'
        if any(ch in value for ch in ":#[]{}&*!|>%@`'\""):
            return json.dumps(value, ensure_ascii=False)
        return value
    raise TypeError(type(value))


def dump_multiseed_yaml(raw: dict) -> str:
    lines = [
        "# Generated by the V6 multi-seed notebook.",
        "# Training seeds are 1, 100, 2027. The PerceptionTest split is the canonical seed-42 split.",
    ]
    for key, value in raw.items():
        if isinstance(value, dict):
            lines.append(f"{key}:")
            for sub, subval in value.items():
                lines.append(f"  {sub}: {_yaml_scalar(subval)}")
        elif isinstance(value, list):
            inner = ", ".join(_yaml_scalar(item) for item in value)
            lines.append(f"{key}: [{inner}]")
        else:
            lines.append(f"{key}: {_yaml_scalar(value)}")
    return "\n".join(lines) + "\n"


def write_multiseed_yaml(path: Path, *, seed: int, output_root: Path, paths: dict[str, str]) -> None:
    import sys
    sys.path.insert(0, str(ROOT))
    from ablation.config import load_yaml

    raw = load_yaml(ROOT / "configs" / "ovis_multiseed.yaml")
    raw["seed"] = int(seed)
    raw["variants"] = ["global_only", "local_only", "hybrid"]
    raw["output_root"] = str(output_root)
    merged = dict(raw.get("paths") or {})
    merged.update(paths)
    raw["paths"] = merged
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dump_multiseed_yaml(raw), encoding="utf-8")
    print("Wrote", path, {"seed": seed, "output_root": str(output_root)})

canon_yaml = CONFIG_DIR / "canonical_split.yaml"
write_multiseed_yaml(
    canon_yaml,
    seed=SPLIT_SEED,
    output_root=CANON,
    paths={
        "train_manifest": "",
        "val_manifest": "",
        "model_dir": "/content/checkpoints/ovis2_5_9b",
        "videos_dir": "/content/perceptiontest_lite/videos",
    },
)

def run(stage: str, config: Path, seed: int) -> None:
    cmd = [str(PYTHON), "run_ablation.py", stage, "--config", str(config), "--seed", str(seed)]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(ROOT), check=True)

run("prepare", canon_yaml, SPLIT_SEED)
run("unit", canon_yaml, SPLIT_SEED)

train_manifest = CANON / "data" / "train_manifest.json"
val_manifest = CANON / "data" / "val_manifest.json"
assert train_manifest.is_file() and val_manifest.is_file(), "prepare did not write the canonical split"
train_payload = json.loads(train_manifest.read_text(encoding="utf-8"))
val_payload = json.loads(val_manifest.read_text(encoding="utf-8"))
train_ids = [str(sample["sample_id"]) for sample in train_payload["samples"]]
val_ids = [str(sample["sample_id"]) for sample in val_payload["samples"]]
train_videos = set(train_payload["video_ids"])
val_videos = set(val_payload["video_ids"])
assert len(train_ids) == 454 and len(train_videos) == 128
assert len(val_ids) == 67 and len(val_videos) == 19
assert len(set(train_ids)) == 454 and len(set(val_ids)) == 67
assert set(train_ids).isdisjoint(val_ids)
assert train_videos.isdisjoint(val_videos)

meta = json.loads((CANON / "shared" / "schedule_meta.json").read_text(encoding="utf-8"))
assert int(meta["seed"]) == SPLIT_SEED
assert meta["train_questions"] == 454 and meta["train_videos"] == 128
assert meta["val_questions"] == 67 and meta["val_videos"] == 19
report = json.loads((CANON / "unit" / "invariant_report.json").read_text(encoding="utf-8"))
assert report["ok"] is True
print("Canonical split OK", {"schedule": meta["schedule_sha256"][:12], "token_mlp": report["token_mlp"]})

persist = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
if persist.parent.exists() and persist.resolve() != CANON.resolve():
    import shutil
    persist.mkdir(parents=True, exist_ok=True)
    shutil.copytree(CANON, persist / "canonical_split", dirs_exist_ok=True)
    print("Copied canonical split →", persist / "canonical_split")


## 7. Train seed 1, rồi 100, rồi 2027

Mỗi seed: `prepare` riêng, assert lịch đúng seed và cùng sample id với split seed 42, rồi train ba biến thể trong một process. Smoke hai update của seed 1 chạy trước train nếu chưa có checkpoint. Xong seed nào thì copy cả thư mục seed lên Drive.


In [ ]:
# 7. Prepare and train global_only, local_only, hybrid for seeds 1, 100, 2027.
from __future__ import annotations
import json
import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
CANON = OUT_ROOT / "canonical_split"
CONFIG_DIR = ROOT / "configs" / "multiseed"
assert PYTHON.exists(), "Chạy cell cài ovis_env trước"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("RELJOIN_ROOT", str(ROOT))
os.environ.setdefault("OVIS_ABLATION_ROOT", str(ROOT))

try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]
try:
    VARIANT
except NameError:
    VARIANT = "global_only,local_only,hybrid"
try:
    VARIANT_LIST
except NameError:
    VARIANT_LIST = ["global_only", "local_only", "hybrid"]
try:
    PERSIST
except NameError:
    PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
    if not PERSIST.parent.exists():
        PERSIST = OUT_ROOT / "persist_local"

def _yaml_scalar(value):
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, float):
        if value.is_integer():
            return f"{value:.1f}"
        return format(value, ".12g")
    if isinstance(value, int):
        return str(value)
    if isinstance(value, str):
        if value == "":
            return '""'
        if any(ch in value for ch in ":#[]{}&*!|>%@`'\""):
            return json.dumps(value, ensure_ascii=False)
        return value
    raise TypeError(type(value))


def dump_multiseed_yaml(raw: dict) -> str:
    lines = [
        "# Generated by the V6 multi-seed notebook.",
        "# Training seeds are 1, 100, 2027. The PerceptionTest split is the canonical seed-42 split.",
    ]
    for key, value in raw.items():
        if isinstance(value, dict):
            lines.append(f"{key}:")
            for sub, subval in value.items():
                lines.append(f"  {sub}: {_yaml_scalar(subval)}")
        elif isinstance(value, list):
            inner = ", ".join(_yaml_scalar(item) for item in value)
            lines.append(f"{key}: [{inner}]")
        else:
            lines.append(f"{key}: {_yaml_scalar(value)}")
    return "\n".join(lines) + "\n"


def write_multiseed_yaml(path: Path, *, seed: int, output_root: Path, paths: dict[str, str]) -> None:
    import sys
    sys.path.insert(0, str(ROOT))
    from ablation.config import load_yaml

    raw = load_yaml(ROOT / "configs" / "ovis_multiseed.yaml")
    raw["seed"] = int(seed)
    raw["variants"] = ["global_only", "local_only", "hybrid"]
    raw["output_root"] = str(output_root)
    merged = dict(raw.get("paths") or {})
    merged.update(paths)
    raw["paths"] = merged
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dump_multiseed_yaml(raw), encoding="utf-8")
    print("Wrote", path, {"seed": seed, "output_root": str(output_root)})

train_manifest = CANON / "data" / "train_manifest.json"
val_manifest = CANON / "data" / "val_manifest.json"
assert train_manifest.is_file() and val_manifest.is_file(), "Chạy cell canonical split trước"
canon_train_ids = [str(sample["sample_id"]) for sample in json.loads(train_manifest.read_text(encoding="utf-8"))["samples"]]
canon_val_ids = [str(sample["sample_id"]) for sample in json.loads(val_manifest.read_text(encoding="utf-8"))["samples"]]
assert len(canon_train_ids) == 454 and len(canon_val_ids) == 67

def seed_paths() -> dict[str, str]:
    return {
        "train_manifest": str(train_manifest.resolve()),
        "val_manifest": str(val_manifest.resolve()),
        "model_dir": "/content/checkpoints/ovis2_5_9b",
        "videos_dir": "/content/perceptiontest_lite/videos",
    }

def run(args: list[str]) -> None:
    cmd = [str(PYTHON), "run_ablation.py", *args]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(ROOT), check=True)

def publish(seed_out: Path, seed: int) -> None:
    if not PERSIST.parent.exists() and PERSIST.name != "persist_local":
        print("Drive not mounted; skip publish", seed)
        return
    dest = PERSIST / f"seed_{seed}"
    if dest.resolve() == seed_out.resolve():
        print("Persist is the local seed dir; skip copy", seed)
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(seed_out, dest, dirs_exist_ok=True)
    print("Copied", seed_out, "→", dest)

hashes = {}
for seed in SEEDS:
    seed_out = OUT_ROOT / f"seed_{seed}"
    cfg_path = CONFIG_DIR / f"seed_{seed}.yaml"
    write_multiseed_yaml(cfg_path, seed=seed, output_root=seed_out, paths=seed_paths())
    run(["prepare", "--config", str(cfg_path), "--seed", str(seed)])
    meta = json.loads((seed_out / "shared" / "schedule_meta.json").read_text(encoding="utf-8"))
    assert int(meta["seed"]) == int(seed), meta["seed"]
    assert meta["train_questions"] == 454 and meta["train_videos"] == 128
    assert meta["val_questions"] == 67 and meta["val_videos"] == 19
    assert Path(meta["train_manifest"]).resolve() == train_manifest.resolve()
    assert Path(meta["val_manifest"]).resolve() == val_manifest.resolve()
    assert meta["frames"]["probed"] is True
    assert int(meta["frames"]["rows_ready"]) == int(meta["frames"]["rows_total"])
    missing_videos = meta["frames"].get("missing_videos") or {}
    assert not missing_videos, f"seed {seed} missing videos: {list(missing_videos)[:8]}"
    epoch_rows = []
    for line in (seed_out / "shared" / "train_schedule.jsonl").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if int(row["epoch"]) == 1:
            epoch_rows.append(str(row["sample_id"]))
    assert epoch_rows == canon_train_ids, f"seed {seed} train schedule is not the canonical sample order"
    val_rows = json.loads((seed_out / "shared" / "val_schedule.json").read_text(encoding="utf-8"))
    assert [str(row["sample_id"]) for row in val_rows] == canon_val_ids
    hashes[seed] = meta["schedule_sha256"]
    print({"seed": seed, "schedule": meta["schedule_sha256"][:12]})

    checkpoints = [seed_out / f"seed_{seed}" / name / "best" / "adapter.pt" for name in VARIANT_LIST]
    if seed == SEEDS[0] and not all(path.is_file() for path in checkpoints):
        run(["smoke", "--config", str(cfg_path), "--variant", VARIANT, "--seed", str(seed)])
        report_path = seed_out / "smoke" / "smoke_report.json"
        report = json.loads(report_path.read_text(encoding="utf-8"))
        if report.get("status") != "ok" or report.get("paper_result") is not False:
            raise RuntimeError("Smoke failed; refusing full training.\n" + json.dumps(report, indent=2))
        print("Smoke passed for seed", seed, "— not a paper result")
    elif seed == SEEDS[0]:
        print("Seed 1 already has best checkpoints; skip smoke")

    run(["train", "--config", str(cfg_path), "--variant", VARIANT, "--seed", str(seed), "--resume"])
    missing_ckpts = [str(path) for path in checkpoints if not path.is_file()]
    if missing_ckpts:
        raise FileNotFoundError("Train did not write best checkpoints:\n" + "\n".join(missing_ckpts))
    for name in VARIANT_LIST:
        selected = json.loads((seed_out / f"seed_{seed}" / name / "selected.json").read_text(encoding="utf-8"))
        print({"seed": seed, "variant": name, "epoch": selected.get("epoch"), "val_loss": selected.get("val_loss")})
    publish(seed_out, seed)

assert len(set(hashes.values())) == len(SEEDS), f"Training schedules did not change across seeds: {hashes}"
print("Train loop finished", {seed: digest[:12] for seed, digest in hashes.items()})


## 8. Validation $L$ and train-loss curves


In [ ]:
# 8. Plot validation L and train loss for every seed and variant.
from __future__ import annotations
import json
from pathlib import Path

ROOT = Path("/content")
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]
try:
    VARIANT_LIST
except NameError:
    VARIANT_LIST = ["global_only", "local_only", "hybrid"]

found = False
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for seed in SEEDS:
    for name in VARIANT_LIST:
        variant_dir = OUT_ROOT / f"seed_{seed}" / f"seed_{seed}" / name
        label = f"{seed} {name}"
        val_path = variant_dir / "val.jsonl"
        if val_path.is_file():
            rows = [json.loads(line) for line in val_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            axes[0].plot([row["epoch"] for row in rows], [row["val_loss"] for row in rows], marker="o", label=label)
            found = True
        train_path = variant_dir / "train.jsonl"
        if train_path.is_file():
            rows = [json.loads(line) for line in train_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            updates = [row for row in rows if row.get("event") == "update"]
            axes[1].plot([row["global_step"] for row in updates], [row["loss"] for row in updates], label=label)
            found = True
if not found:
    print("No seed outputs yet. Run the train cell first.")
else:
    axes[0].set_title("validation L")
    axes[0].set_xlabel("epoch")
    axes[1].set_title("train loss per update")
    axes[1].set_xlabel("global step")
    axes[0].legend(fontsize=8)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    dest = OUT_ROOT / "loss_curves.png"
    dest.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(dest, dpi=120)
    print("Wrote", dest)
    plt.show()


## 9. Export artifacts


In [ ]:
# 9. Copy the multiseed tree to Drive and zip it.
from __future__ import annotations
import shutil
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/content")
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
persist = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
if not persist.parent.exists():
    persist = OUT_ROOT / "persist_local"
persist.mkdir(parents=True, exist_ok=True)
if OUT_ROOT.exists() and persist.resolve() != OUT_ROOT.resolve():
    shutil.copytree(OUT_ROOT, persist, dirs_exist_ok=True)
    print("Copied", OUT_ROOT, "→", persist)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"frameweave_ovis_multiseed_train_{stamp}"
if OUT_ROOT.exists():
    shutil.make_archive(str(zip_path), "zip", root_dir=OUT_ROOT)
    print("Zip:", f"{zip_path}.zip")
print("Persist:", persist)
